## Setup

In [ ]:
import os
import sys

import pandas as pd
from dotenv import load_dotenv

sys.path.append("../src")

import utils
import backends

# NOTE: define variables in .env to avoid changing the notebook
load_dotenv("../.env")

In [ ]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
BACKEND = os.getenv("BACKEND", "gemini")
API_KEY = os.getenv("API_KEY")

print(f"{FILENAME=}")
print(f"{BACKEND=}")
print(f"{API_KEY=}")

# TODO: generalize for any backend, use env vars
backend = backends.get(api_key=API_KEY, backend=BACKEND)
df = pd.read_csv(f"../data/raw/{FILENAME}.csv")

print(f"{df.shape=}")
df.head(3)

## Preprocess

In [ ]:
# Ensure 'property' column is string type
df["property"] = df["property"].astype(str)

# Remove trailing spaces from 'property' column
df["property"] = df["property"].str.rstrip()

# Extract unique properties and embed them
props = df["property"].dropna().unique()
props_embeds = utils.embed(backend, pd.Series(props), batch_size=100)

# Map properties to their embeddings
props2vec = dict(zip(props, props_embeds))
df["prop_embedding"] = df["property"].map(props2vec)

# Create cumulative concatenations of properties
grouped = df.groupby(["id", "category", "concept"])
df["properties_cum"] = grouped["property"].transform(utils.cumulative_concat)

df.head(3)

## Embed

In [ ]:
# Embed cumulatively concatenated strings
df["embedding"] = utils.embed(backend, df["properties_cum"], batch_size=100)
df.head(3)

In [ ]:
# Save embeddings
os.makedirs("../data/embeddings/", exist_ok=True)
utils.save(df, f"../data/embeddings/{FILENAME}.csv")
print(f"{df.shape=}")